# Flattening the image annotations
Assuming step one was completed, the following steps will extract the bounding boxes and visual attributes from the database created in step 1.
1. Import the pickle file.
2. Create pydantic models to ensure proper formating.
3. Iterate over the pkl file to flatten the attributes accordingly.

In [219]:
import pickle

# Open the file in binary mode for reading
with open('annotations/pie_database.pkl', 'rb') as file:
    data = pickle.load(file)


In [ ]:
from pydantic import BaseModel

class BBox(BaseModel):
    video: str
    frame: int
    id: str
    occlusion: int
    
    x1: float
    y1: float
    x2: float
    y2: float

    content: str = 'pedestrian'
    state: int = None

    action: int = None
    gesture: int = None
    cross: int = None
    look: int = None
    
class Pedestrian(BaseModel):
    id: str
    age: int
    critical_point: int
    crossing: int
    crossing_point: int
    exp_start_point: int
    gender: int
    intention_prob: float
    intersection: int
    num_lanes: int
    signalized: int
    traffic_direction: int

class Vehicle(BaseModel):
    video: str
    frame: int
    GPS_speed: float
    OBD_speed: float
    accX: float
    accY: float
    accZ: float
    gyroX: float
    gyroY: float
    gyroZ: float
    heading_angle: float
    latitude: float
    longitude: float
    pitch: float
    roll: float
    yaw: float

In [216]:
bboxes = []
pedestrians = []
vehicles = []

for video_id, video_data in data['set01'].items():
    for id, vehicle in video_data['vehicle_annotations'].items():
        veh = Vehicle(
            video=video_id,
            frame=id,
            GPS_speed=vehicle['GPS_speed'],
            OBD_speed=vehicle['OBD_speed'],
            accX=vehicle['accX'],
            accY=vehicle['accY'],
            accZ=vehicle['accZ'],
            gyroX=vehicle['gyroX'],
            gyroY=vehicle['gyroY'],
            gyroZ=vehicle['gyroZ'],
            heading_angle=vehicle['heading_angle'],
            latitude=vehicle['latitude'],
            longitude=vehicle['longitude'],
            pitch=vehicle['pitch'],
            roll=vehicle['roll'],
            yaw=vehicle['yaw']
        )
        vehicles.append(veh.model_dump())
    
    for id, traffic in video_data['traffic_annotations'].items():
        frames = traffic['frames']
        bboxs = traffic['bbox']
        occlusions = traffic['occlusion']
        content = traffic['obj_class']
        obj_type = traffic['obj_type']
        states = traffic['state']
 
        for frame, box, occlusion, state in zip(frames, bboxs, occlusions, states):
            obj = BBox(
                video=video_id,
                id=id,
                frame=frame,
                occlusion=occlusion,
                x1=box[0],
                y1=box[1],
                x2=box[2],
                y2=box[3],
                content=content,
                state=state
            ) 
            bboxes.append(obj.model_dump())

    for id, ped in video_data['ped_annotations'].items():
        pedestrian = ped['attributes']
        pedestrian['id'] = id 
        pedestrians.append(Pedestrian(**pedestrian).model_dump())
        
        frames = ped['frames']
        bboxs = ped['bbox']
        occlusions = ped['occlusion']
        behaviors = ped['behavior']

        for frame, box, occlusion, gesture, look, action, cross in zip(frames, bboxs, occlusions, *behaviors.values()):
            bbox = BBox(
                video=video_id,
                frame=frame,
                id=id,
                occlusion=occlusion,
                x1=box[0],
                y1=box[1],
                x2=box[2],
                y2=box[3],
                gesture=gesture,
                look=look,
                action=action,
                cross=cross
            ).model_dump()
            bboxes.append(bbox)

import pandas as pd
pedestrians_df = pd.DataFrame(pedestrians)
bboxes_df      = pd.DataFrame(bboxes)
vehicles_df    = pd.DataFrame(vehicles)

idmap = {id: index for index, id in enumerate(bboxes_df.id.unique()) }

bboxes_df['id'] = bboxes_df['id'].map(idmap)
pedestrians_df['id'] = pedestrians_df['id'].map(idmap)

In [217]:
pedestrians_df.to_csv('annotations/pedestrians.csv', index=False)
bboxes_df.to_csv('annotations/bboxes.csv', index=False)
vehicles_df.to_csv('annotations/vehicles.csv', index=False)